In [ ]:
from pixell import enmap, enplot
from pspy import so_map_preprocessing, pspy_utils
from mnms import utils

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# multiply wmask by tapered edge at RA=10
geometry = enmap.read_map_geometry('/home/zatkins/scratch/projects/lat-iso/piso/tiger_dr6xdeep56_20251019/windows/window_so_i1_f090_baseline.fits')
emask = enmap.zeros(*geometry, np.float32)
_, pix_x = enmap.sky2pix(*emask.geometry, np.deg2rad([[0], [10]]))
pix_x = np.round(pix_x.squeeze()).astype(int)
emask[:, :pix_x] = 1
emask = enmap.apod_mask(emask, width=np.deg2rad(2), edge=False)

In [ ]:
# since the 1d spectra aren't super useful, this does the same as above but
# just for SO 2d spectra. also we save compute by skipping pol crosses

mask_dir = '/home/zatkins/scratch/projects/lat-iso/piso/tiger_dr6xdeep56_20251019/windows'

mapset = 'so_i5_f280'

if mapset[:2] == 'so':
    map_dir = '/home/zatkins/scratch/projects/lat-iso/piso/tiger_dr6xdeep56_20251019/maps/so'
    maps = [enmap.read_map(f'{map_dir}/sky_map_{mapset[3:]}_set{i}_20251019.fits') for i in range(4)]
else:
    map_dir = '/home/zatkins/scratch/projects/lat-iso/piso/tiger_dr6xdeep56_20251019/maps/act_extracted'
    maps = [enmap.read_map(f'{map_dir}/act_dr6.02_std_AA_night_{mapset[4:]}_4way_set{i}_map.fits') for i in range(4)]

maps = enmap.samewcs(maps, maps[0])
wmask = enmap.read_map(f'{mask_dir}/window_{mapset}_baseline.fits')

# for s in range(4):
#     for p in range(3):
#         enplot.pshow(maps[s, p], colorbar=True, ticks=5, downgrade=16, range=[600, 300, 300][p])
#         enplot.pshow(maps[s, p] * wmask * emask, colorbar=True, ticks=5, downgrade=16, range=[600, 300, 300][p])

_bin_edges = np.arange(0, 20000, 50) 
modlmap = maps.modlmap().astype(np.float32)
lwcs = enmap.lwcs(maps.shape, maps.wcs)

kpols = 'IQU'
kspecs_2d = {}
nspecs_2d = {}

kmaps = enmap.fft(maps * wmask * emask)
kmaps_mean = kmaps.mean(axis=0) # mean over splits

for x in range(3):
    for y in range(x, 3):
        if x != y:
            continue
        
        spec = kpols[x] + kpols[y]
        kspecs_2d[spec] = 0
        nspecs_2d[spec] = 0

        for i in range(4):
            nspecs_2d[spec] += np.real((kmaps[i][x] - kmaps_mean[x]) * (kmaps[i][y] - kmaps_mean[y]).conj())

            for j in range(i+1, 4):
                kspecs_2d[spec] += np.real(kmaps[i][x] * kmaps[j][y].conj())
        
        kspecs_2d[spec] /= utils.interp1d_bins(_bin_edges, utils.radial_bin(kspecs_2d[spec], modlmap, _bin_edges), bounds_error=False)(modlmap)
        kspecs_2d[spec] = enmap.ndmap(kspecs_2d[spec], lwcs)

        nspecs_2d[spec] /= utils.interp1d_bins(_bin_edges, utils.radial_bin(nspecs_2d[spec], modlmap, _bin_edges), bounds_error=False)(modlmap)
        nspecs_2d[spec] = enmap.ndmap(nspecs_2d[spec], lwcs)

enplot.pshow(utils.crop_center(enmap.fftshift(kspecs_2d['II']), 100, 400), ticks=[100, 180], colorbar=True, upgrade=(4, 1), min=-4, max=6)
enplot.pshow(utils.crop_center(enmap.fftshift(kspecs_2d['QQ']), 100, 400), ticks=[100, 180], colorbar=True, upgrade=(4, 1), min=-4, max=6)
enplot.pshow(utils.crop_center(enmap.fftshift(kspecs_2d['UU']), 100, 400), ticks=[100, 180], colorbar=True, upgrade=(4, 1), min=-4, max=6)

enplot.pshow(utils.crop_center(enmap.fftshift(kspecs_2d['II']).downgrade(4), 100, 400), ticks=[400, 720], colorbar=True, upgrade=(4, 1), min=-2, max=4)
enplot.pshow(utils.crop_center(enmap.fftshift(kspecs_2d['QQ']).downgrade(4), 100, 400), ticks=[400, 720], colorbar=True, upgrade=(4, 1), min=-2, max=4)
enplot.pshow(utils.crop_center(enmap.fftshift(kspecs_2d['UU']).downgrade(4), 100, 400), ticks=[400, 720], colorbar=True, upgrade=(4, 1), min=-2, max=4)

enplot.pshow(utils.crop_center(enmap.fftshift(nspecs_2d['II']), 100, 400), ticks=[100, 180], colorbar=True, upgrade=(4, 1), min=-4, max=6)
enplot.pshow(utils.crop_center(enmap.fftshift(nspecs_2d['QQ']), 100, 400), ticks=[100, 180], colorbar=True, upgrade=(4, 1), min=-4, max=6)
enplot.pshow(utils.crop_center(enmap.fftshift(nspecs_2d['UU']), 100, 400), ticks=[100, 180], colorbar=True, upgrade=(4, 1), min=-4, max=6)

enplot.pshow(utils.crop_center(enmap.fftshift(nspecs_2d['II']).downgrade(4), 100, 400), ticks=[400, 720], colorbar=True, upgrade=(4, 1), min=-2, max=4)
enplot.pshow(utils.crop_center(enmap.fftshift(nspecs_2d['QQ']).downgrade(4), 100, 400), ticks=[400, 720], colorbar=True, upgrade=(4, 1), min=-2, max=4)
enplot.pshow(utils.crop_center(enmap.fftshift(nspecs_2d['UU']).downgrade(4), 100, 400), ticks=[400, 720], colorbar=True, upgrade=(4, 1), min=-2, max=4)